In [8]:
# Importa bibliotecas necessárias
import pandas as pd
import numpy as np
import time
import tensorflow as tf

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout

In [9]:
# Definindo os caminho da nova pasta no PC pessoal
# O caminho usa '../../' pois o notebook está em scripts/Redução de Dimensionalidade/
caminho_pasta_tratado = '../../dataset tratado/cicids2017/'

nome_dados_treinamento = 'cicids2017_treinamento.csv'
nome_dados_teste = 'cicids2017_teste.csv'

nome_dados_treinamento_reduzidos = 'Redução de Dimensionalidade/cicids2017_treinamento_reduzidos.csv'
nome_dados_teste_reduzidos = 'Redução de Dimensionalidade/cicids2017_teste_reduzidos.csv'

In [15]:
# 1. Carregando o input a partir dos CSVs de dados JÁ TRATADOS
print("Carregando os datasets tratados em CSV...")
df_treino = pd.read_csv(caminho_pasta_tratado + nome_dados_treinamento)
df_teste = pd.read_csv(caminho_pasta_tratado + nome_dados_teste)

print(f"Dados carregados! Treino: {df_treino.shape} | Teste: {df_teste.shape}")

Carregando os datasets tratados em CSV...
Dados carregados! Treino: (1979513, 79) | Teste: (848363, 79)


In [16]:
# 2. Removendo características identificadoras/enviesadas antes da seleção por MDI
# No dataset tratado atual, o campo identificador/enviesado ainda presente é a porta de destino.
# A coluna duplicada de Fwd Header Length também é removida para reduzir redundância.
nomes_enviesados = {
    ' Flow ID',
    ' Source IP',
    ' Source Port',
    ' Destination IP',
    ' Destination Port',
    ' Protocol',
    ' Timestamp',
    ' Fwd Header Length.1'
}

colunas_para_remover = []
fwd_header_length_vistos = 0

for coluna in df_treino.columns:
    nome_normalizado = coluna.strip().lower()

    if coluna in nomes_enviesados:
        colunas_para_remover.append(coluna)

    if nome_normalizado == 'fwd header length':
        fwd_header_length_vistos += 1
        if fwd_header_length_vistos > 1:
            colunas_para_remover.append(coluna)

    if nome_normalizado.startswith('fwd header length.'):
        colunas_para_remover.append(coluna)

colunas_para_remover = [coluna for coluna in dict.fromkeys(colunas_para_remover) if coluna != 'Label']
colunas_treino_existentes = [coluna for coluna in colunas_para_remover if coluna in df_treino.columns]
colunas_teste_existentes = [coluna for coluna in colunas_para_remover if coluna in df_teste.columns]

df_treino = df_treino.drop(columns=colunas_treino_existentes)
df_teste = df_teste.drop(columns=colunas_teste_existentes)

print('Características removidas para mitigar overfitting:', colunas_treino_existentes)
print(f'Dimensões após remoção - Treino: {df_treino.shape} | Teste: {df_teste.shape}')

# 3. Separando as características (X) do gabarito/rótulo (y)
X_treino = df_treino.drop('Label', axis=1)
y_treino = df_treino['Label']

X_teste = df_teste.drop('Label', axis=1)
y_teste = df_teste['Label']

Características removidas para mitigar overfitting: [' Destination Port', ' Fwd Header Length.1']
Dimensões após remoção - Treino: (1979513, 77) | Teste: (848363, 77)


In [17]:
# 4. Aplicando Seleção de Características MDI (Mean Decrease in Impurity) com Random Forest
print("\nIniciando o treinamento do Random Forest para extração de features (MDI)...")
rf_fs = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)

inicio_fs = time.time()
rf_fs.fit(X_treino, y_treino)
fim_fs = time.time()

print(f"Treinamento RF concluído! Tempo extração: {(fim_fs - inicio_fs):.4f} segundos.")

importancias = rf_fs.feature_importances_
features = X_treino.columns

df_importancias = pd.DataFrame({'Feature': features, 'Importancia': importancias})
df_importancias = df_importancias.sort_values(by='Importancia', ascending=False)

print("\nQuantidade total de features:", len(features))

# Selecionando as 51 features mais importantes (conforme sugerido na literatura por Neto e Gomes)
top_51_features = df_importancias.head(51)['Feature'].tolist()

quantidade_de_features_descartadas = len(features) - len(top_51_features)
print("Quantidade de features selecionadas (top 51):", len(top_51_features))
print("Quantidade de features descartadas:", quantidade_de_features_descartadas)

# Mostra qual foi o corte de importância para as features selecionadas
importancia_corte = df_importancias[df_importancias['Feature'] == top_51_features[-1]]['Importancia'].values[0]
print(f"Importância mínima para entrar no top 51: {importancia_corte:.6f}")
display(df_importancias.tail(quantidade_de_features_descartadas + 5))

print("\nTop 10 features mais importantes:")
display(df_importancias.head(10))


Iniciando o treinamento do Random Forest para extração de features (MDI)...
Treinamento RF concluído! Tempo extração: 65.2246 segundos.

Quantidade total de features: 76
Quantidade de features selecionadas (top 51): 51
Quantidade de features descartadas: 25
Importância mínima para entrar no top 51: 0.002621


,Feature,Importancia
27,Bwd IAT Max,3.438520e-03
68,Active Mean,2.920115e-03
25,Bwd IAT Mean,2.740683e-03
71,Active Min,2.706856e-03
24,Bwd IAT Total,2.620871e-03
18,Flow IAT Min,2.527920e-03
4,Total Length of Bwd Packets,2.514892e-03
26,Bwd IAT Std,2.154481e-03
42,FIN Flag Count,1.945748e-03
28,Bwd IAT Min,1.732957e-03



Top 10 features mais importantes:


,Feature,Importancia
11,Bwd Packet Length Mean,0.055883
51,Average Packet Size,0.053185
39,Packet Length Mean,0.051992
12,Bwd Packet Length Std,0.048513
41,Packet Length Variance,0.044265
64,Init_Win_bytes_forward,0.043264
40,Packet Length Std,0.042768
9,Bwd Packet Length Max,0.040962
3,Total Length of Fwd Packets,0.037278
53,Avg Bwd Segment Size,0.035490


In [19]:
# 5. Reduzindo a dimensionalidade dos conjuntos de treino e teste
X_treino_reduzido = X_treino[top_51_features]
X_teste_reduzido = X_teste[top_51_features]

print(f"Novas dimensões - Treino: {X_treino_reduzido.shape} | Teste: {X_teste_reduzido.shape}")

# Salvando os datasets reduzidos em CSV para uso posterior
X_treino_reduzido.to_csv(caminho_pasta_tratado + nome_dados_treinamento_reduzidos, index=False)
print(f"Dataset de treino reduzido salvo em: {caminho_pasta_tratado + nome_dados_treinamento_reduzidos}")

X_teste_reduzido.to_csv(caminho_pasta_tratado + nome_dados_teste_reduzidos, index=False)
print(f"Dataset de teste reduzido salvo em: {caminho_pasta_tratado + nome_dados_teste_reduzidos}")


Novas dimensões - Treino: (1979513, 51) | Teste: (848363, 51)
Dataset de treino reduzido salvo em: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_treinamento_reduzidos.csv
Dataset de teste reduzido salvo em: ../../dataset tratado/cicids2017/Redução de Dimensionalidade/cicids2017_teste_reduzidos.csv


In [ ]:
# 6. Convertendo as Labels de Texto para Números
print("\nCodificando as labels para a Rede Neural...")
encoder = LabelEncoder()
y_treino_num = encoder.fit_transform(y_treino)
y_teste_num = encoder.transform(y_teste)
num_classes = len(encoder.classes_)

In [ ]:
# 7. Construindo a Arquitetura da Rede Neural Profunda (DNN) com a entrada reduzida
dnn_model = Sequential([
    keras.layers.Input(shape=(X_treino_reduzido.shape[1],)),
    Dense(128, activation='relu'),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

dnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

In [ ]:
# 8. O Treinamento (Medindo o Custo Computacional)
print("\nIniciando o treinamento da Rede Neural Profunda (DNN) com dimensionalidade reduzida...")
inicio_treino_dnn = time.time()

history = dnn_model.fit(X_treino_reduzido, y_treino_num, epochs=10, batch_size=256, validation_split=0.1)

fim_treino_dnn = time.time()
tempo_treinamento_dnn = fim_treino_dnn - inicio_treino_dnn
print(f"\nTreinamento DNN concluído! Tempo total: {tempo_treinamento_dnn:.4f} segundos.")

In [ ]:
# 9. Classificação dos Dados de Teste
print("\nIniciando a classificação dos dados de teste...")
inicio_teste_dnn = time.time()

previsoes_probabilidades = dnn_model.predict(X_teste_reduzido)
previsoes_dnn_num = np.argmax(previsoes_probabilidades, axis=1)

fim_teste_dnn = time.time()
tempo_teste_dnn = fim_teste_dnn - inicio_teste_dnn
print(f"Classificação concluída! Tempo de previsão: {tempo_teste_dnn:.4f} segundos.")

previsoes_dnn_labels = encoder.inverse_transform(previsoes_dnn_num)

# 10. Avaliação dos Resultados
print("\n=== MATRIZ DE CONFUSÃO (DNN - Reduzida) ===")
labels = sorted(np.unique(np.concatenate([y_teste.values, previsoes_dnn_labels])))
cm = confusion_matrix(y_teste, previsoes_dnn_labels, labels=labels)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{lbl}" for lbl in labels],
    columns=[f"Pred_{lbl}" for lbl in labels]
)

def color_confusion_matrix(data):
    styles = pd.DataFrame("", index=data.index, columns=data.columns)
    for i in range(min(data.shape[0], data.shape[1])):
        styles.iat[i, i] = "background-color: #c6efce; color: #006100; font-weight: bold;"
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if i != j and data.iat[i, j] != 0:
                styles.iat[i, j] = "background-color: #ffc7ce; color: #9c0006; font-weight: bold;"
    return styles

display(cm_df.style.apply(color_confusion_matrix, axis=None).format("{:.0f}"))

print("\n=== RELATÓRIO DE MÉTRICAS (Precisão, Recall, F1-Score) ===")
print(classification_report(y_teste, previsoes_dnn_labels, zero_division=0))